In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Help Functions

In [ ]:
# Functions that will help cleaning the dataset
def get_winner(df, cat=False):
    """
    Get the winner from the dataset based on goals
    """
    if df['home_team_score'] > df['away_team_score']:
        return df['home_team'] if not cat else 1
    return df['away_team'] if not cat else 0

def clean_stage(df):
    """
    Remove the groups x and replace it just with Groups
    """
    if df['Stage'].__contains__('Group'):
        return 'Group'
    return df['Stage']

def mix_fifa_data(df, fifa_data):
    """
    Add fifa data
    """
    df.reset_index(drop=True, inplace=True)
    for i in range(df.shape[0]):
        try:
            df['home_overall'][i] = fifa_data[fifa_data['Nationality'] == df['home_team'][i]]['Overall'].values[0].astype(np.float64)
            df['away_overall'][i] = fifa_data[fifa_data['Nationality'] == df['away_team'][i]]['Overall'].values[0].astype(np.float64)
            df['home_potential'][i] = fifa_data[fifa_data['Nationality'] == df['home_team'][i]]['Potential'].values[0].astype(np.float64)
            df['away_potential'][i] = fifa_data[fifa_data['Nationality'] == df['home_team'][i]]['Potential'].values[0].astype(np.float64)
            df['home_reputation'][i] = fifa_data[fifa_data['Nationality'] == df['home_team'][i]]['International Reputation'].values[0].astype(np.float64)
            df['away_reputation'][i] = fifa_data[fifa_data['Nationality'] == df['home_team'][i]]['International Reputation'].values[0].astype(np.float64)
        except:
            df['home_overall'][i] = 0
            df['away_overall'][i] = 0
            df['home_potential'][i] = 0
            df['away_potential'][i] = 0
            df['home_reputation'][i]= 0
            df['away_reputation'][i]= 0

def get_prediction_df(home_team, away_team):
    """
    Get the prediction df frame from a home_team and away_team
    """
    columns = ['team_fifa_rank', 'team_total_fifa_points', 'overall', 'potential']
    data_dict = {}
    for column in columns:
        data_dict['home_' + column] = country_gb['home_' + column][home_team]
        data_dict['away_' + column] = country_gb['away_' + column][away_team]

    predict_df = pd.DataFrame([data_dict])

    predict_df['overall_difference'] = predict_df['home_overall'] - predict_df['away_overall']
    predict_df['potential_difference'] = predict_df['home_potential'] - predict_df['away_potential']
    predict_df['rank_difference'] = predict_df['home_team_fifa_rank'] - predict_df['away_team_fifa_rank']
    predict_df['average_rank'] = (predict_df['home_team_fifa_rank'] + predict_df['away_team_fifa_rank'])/2
    predict_df['point_difference'] = predict_df['home_team_total_fifa_points'] - predict_df['away_team_total_fifa_points']

    predict_df.drop(['home_team_fifa_rank', 'away_team_fifa_rank',
       'home_team_total_fifa_points', 'away_team_total_fifa_points',
       'home_overall', 'away_overall','home_potential', 'away_potential',
       ], axis=1, inplace=True)
    
    return predict_df

## Start

In [ ]:
# Datos desde el local
international_matches = pd.read_csv("files/international_matches.csv")
data_fifa_23 = pd.read_csv('files/FIFA23_official_data.csv')

In [ ]:
#List of all Teams in 2022 World Cup
teams_worldcup = ['Qatar', 'Ecuador', 'Senegal', 'Netherlands', 'England', 'Iran', 'USA',
                  'Wales', 'Argentina', 'Saudi Arabia', 'Mexico', 'Poland', 'France', 
                  'Australia', 'Denmark', 'Tunisia', 'Spain', 'Costa Rica', 'Germany', 
                  'Japan', 'Belgium', 'Canada', 'Morocco', 'Croatia', 'Brazil', 'Serbia', 
                  'Switzerland', 'Cameroon', 'Portugal', 'Ghana', 'Uruguay', 'Korea Republic']

## Players Data

In [ ]:
# Elimino Columnas a ojo
data_fifa_23.drop(['ID', 'Photo', 'Flag', 'Club', 'Club Logo', 'Release Clause', 'Kit Number', 'Contract Valid Until',
                   'Real Face', 'Body Type', 'Work Rate', 'Position', 'Loaned From', 'Name', 'Preferred Foot', 'Wage', 'Weak Foot', 'Age'], axis=1, inplace=True)

In [ ]:
data_fifa_23.replace({'United States': 'USA'}, inplace=True)

In [ ]:
data_fifa_grouped = data_fifa_23.groupby(['Nationality']).mean()

In [ ]:
data_fifa_grouped.reset_index(inplace=True)

In [ ]:
data_fifa_grouped = data_fifa_grouped.reset_index(level=0)

## Matches International

In [ ]:
# Turn date into a datetime object
international_matches['date'] = pd.to_datetime(international_matches['date'])
international_matches['home_overall'] = np.float64(0)
international_matches['away_overall'] = np.float64(0)
international_matches['home_potential'] = np.float64(0)
international_matches['away_potential'] = np.float64(0)
international_matches['home_reputation'] = np.float64(0)
international_matches['away_reputation'] = np.float64(0)

In [ ]:
international_matches = international_matches.loc[(international_matches['home_team'].isin(teams_worldcup) | international_matches['away_team'].isin(teams_worldcup)) & (international_matches['date'] > '2010-01-01')]
international_matches['Won'] = international_matches.apply(lambda row: get_winner(row, cat=True), axis=1)
mix_fifa_data(international_matches, data_fifa_grouped)
international_matches.drop(['home_team_continent', 'date', 'away_team_continent', 'shoot_out', 'home_team_result', 'tournament', 'city', 'neutral_location', 'country'], inplace=True, axis=1)

In [ ]:
# Country group by
country_gb = international_matches.groupby(['home_team']).mean()

In [ ]:
international_matches

In [ ]:
# combine columns to create features
international_matches['overall_difference'] = international_matches['home_overall'] - international_matches['away_overall']
international_matches['potential_difference'] = international_matches['home_potential'] - international_matches['away_potential']
international_matches['rank_difference'] = international_matches['home_team_fifa_rank'] - international_matches['away_team_fifa_rank']
international_matches['average_rank'] = (international_matches['home_team_fifa_rank'] + international_matches['away_team_fifa_rank'])/2
international_matches['point_difference'] = international_matches['home_team_total_fifa_points'] - international_matches['away_team_total_fifa_points']

international_matches.drop(['home_team', 'away_team', 'home_team_fifa_rank', 'away_team_fifa_rank',
       'home_team_total_fifa_points', 'away_team_total_fifa_points',
       'home_team_score', 'away_team_score', 'home_team_goalkeeper_score',
       'away_team_goalkeeper_score', 'home_team_mean_defense_score',
       'home_team_mean_offense_score', 'home_team_mean_midfield_score',
       'away_team_mean_defense_score', 'away_team_mean_offense_score',
       'away_team_mean_midfield_score', 'home_overall', 'away_overall',
       'home_potential', 'away_potential', 'home_reputation',
       'away_reputation'], axis=1, inplace=True)

In [ ]:
international_matches

In [ ]:
international_matches.head()

## Model

In [ ]:
from sklearn import linear_model, svm, tree
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve

In [ ]:
# Get training and testing data
data = international_matches.copy()

X = data.drop(['Won'], axis=1)
y = data['Won']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
scaler = StandardScaler()

svm_model = svm.SVC()
pipe_svm_std = Pipeline(steps=[("scaler", scaler), ("svm_model", svm_model)])

# Parameters of pipelines can be set using '__' separated parameter names:
param_grid = {
    "svm_model__C": (0.5, 1.5, 1, 2, 3)
}
search_svm_std = GridSearchCV(pipe_svm_std, param_grid, cv=3, scoring="f1", refit=True)
search_svm_std.fit(X, y)
print("Best parameter (CV score=%0.3f):" % search_svm_std.best_score_)
print(search_svm_std.best_params_)

fpr, tpr, _ = roc_curve(y_test, search_svm_std.best_estimator_.predict(X_test))
plt.figure(figsize=(15,5))
ax = plt.subplot(1,3,1)
ax.plot([0, 1], [0, 1], 'k--')
ax.plot(fpr, tpr)
ax.set_title('AUC score is {0:0.2}'.format(roc_auc_score(y_test, search_svm_std.best_estimator_.predict(X_test))))
ax.set_aspect(1)

ax = plt.subplot(1,3,2)
cm = confusion_matrix(y_test, search_svm_std.best_estimator_.predict(X_test))
sns.heatmap(cm, cmap='Blues', clim = (0, cm.max()), annot=True, fmt="d") 

ax.set_xlabel('Predicted label')
ax.set_title('Performance on the Test set')

ax = plt.subplot(1,3,3)
cm = confusion_matrix(y_train, search_svm_std.best_estimator_.predict(X_train))
sns.heatmap(cm, cmap='Blues', clim = (0, cm.max()), annot=True, fmt="d") 
ax.set_xlabel('Predicted label')
ax.set_title('Performance on the Training set')


In [ ]:
scaler_pol = PolynomialFeatures()

svm_model = svm.SVC()
pipe_svm_pol = Pipeline(steps=[("scaler_pol", scaler_pol), ("svm_model", svm_model)])

# Parameters of pipelines can be set using '__' separated parameter names:
param_grid = {
    "svm_model__degree": (1, 2, 3),
    "svm_model__kernel": ("poly",),
    "scaler_pol__degree": (1, 2, 3)
}
search_svm_pol = GridSearchCV(pipe_svm_pol, param_grid, cv=3, scoring="f1", refit=True)
search_svm_pol.fit(X, y)
print("Best parameter (CV score=%0.3f):" % search_svm_pol.best_score_)
print(search_svm_pol.best_params_)

fpr, tpr, _ = roc_curve(y_test, search_svm_pol.best_estimator_.predict(X_test))
plt.figure(figsize=(15,5))
ax = plt.subplot(1,3,1)
ax.plot([0, 1], [0, 1], 'k--')
ax.plot(fpr, tpr)
ax.set_title('AUC score is {0:0.2}'.format(roc_auc_score(y_test, search_svm_pol.best_estimator_.predict(X_test))))
ax.set_aspect(1)

ax = plt.subplot(1,3,2)
cm = confusion_matrix(y_test, search_svm_pol.best_estimator_.predict(X_test))
sns.heatmap(cm, cmap='Blues', clim = (0, cm.max()), annot=True, fmt="d") 

ax.set_xlabel('Predicted label')
ax.set_title('Performance on the Test set')

ax = plt.subplot(1,3,3)
cm = confusion_matrix(y_train, search_svm_pol.best_estimator_.predict(X_train))
sns.heatmap(cm, cmap='Blues', clim = (0, cm.max()), annot=True, fmt="d") 
ax.set_xlabel('Predicted label')
ax.set_title('Performance on the Training set')


In [ ]:
scaler = StandardScaler()

logistic = linear_model.LogisticRegression(max_iter=100000, solver='saga', tol=0.1)
pipe_rg_std = Pipeline(steps=[("scaler", scaler), ("logistic", logistic)])

# Parameters of pipelines can be set using '__' separated parameter names:
param_grid = {
    "logistic__penalty": ('l2', 'l1', 'elasticnet'),
    'logistic__C': (1e-5, 1e-4, 1e-3)
}
search_rg_std = GridSearchCV(pipe_rg_std, param_grid, cv=2, refit=True)
search_rg_std.fit(X, y)
print("Best parameter (CV score=%0.3f):" % search_rg_std.best_score_)
print(search_rg_std.best_params_)

fpr, tpr, _ = roc_curve(y_test, search_rg_std.best_estimator_.predict(X_test))
plt.figure(figsize=(15,5))
ax = plt.subplot(1,3,1)
ax.plot([0, 1], [0, 1], 'k--')
ax.plot(fpr, tpr)
ax.set_title('AUC score is {0:0.2}'.format(roc_auc_score(y_test, search_rg_std.best_estimator_.predict(X_test))))
ax.set_aspect(1)

ax = plt.subplot(1,3,2)
cm = confusion_matrix(y_test, search_rg_std.best_estimator_.predict(X_test))
sns.heatmap(cm, cmap='Blues', clim = (0, cm.max()), annot=True, fmt="d") 

ax.set_xlabel('Predicted label')
ax.set_title('Performance on the Test set')

ax = plt.subplot(1,3,3)
cm = confusion_matrix(y_train, search_rg_std.best_estimator_.predict(X_train))
sns.heatmap(cm, cmap='Blues', clim = (0, cm.max()), annot=True, fmt="d") 
ax.set_xlabel('Predicted label')
ax.set_title('Performance on the Training set')

In [ ]:

scaler = PolynomialFeatures()

logistic = linear_model.LogisticRegression(max_iter=100000, solver='saga', tol=0.1)
pipe = Pipeline(steps=[("scaler", scaler), ("logistic", logistic)])

# Parameters of pipelines can be set using '__' separated parameter names:
param_grid = {
    "scaler__degree": (1, 2, 3),
    "logistic__penalty": ('l2', 'l1', 'elasticnet'),
    'logistic__C': (1e-5, 1e-4, 1e-3)
}
search_rg_pol = GridSearchCV(pipe, param_grid, cv=2, refit=True)
search_rg_pol.fit(X, y)
print("Best parameter (CV score=%0.3f):" % search_rg_pol.best_score_)
print(search_rg_pol.best_params_)

fpr, tpr, _ = roc_curve(y_test, search_rg_pol.best_estimator_.predict(X_test))
plt.figure(figsize=(15,5))
ax = plt.subplot(1,3,1)
ax.plot([0, 1], [0, 1], 'k--')
ax.plot(fpr, tpr)
ax.set_title('AUC score is {0:0.2}'.format(roc_auc_score(y_test, search_rg_pol.best_estimator_.predict(X_test))))
ax.set_aspect(1)

ax = plt.subplot(1,3,2)
cm = confusion_matrix(y_test, search_rg_pol.best_estimator_.predict(X_test))
sns.heatmap(cm, cmap='Blues', clim = (0, cm.max()), annot=True, fmt="d") 

ax.set_xlabel('Predicted label')
ax.set_title('Performance on the Test set')

ax = plt.subplot(1,3,3)
cm = confusion_matrix(y_train, search_rg_pol.best_estimator_.predict(X_train))
sns.heatmap(cm, cmap='Blues', clim = (0, cm.max()), annot=True, fmt="d") 
ax.set_xlabel('Predicted label')
ax.set_title('Performance on the Training set')

## Predict

In [ ]:
while True:
    home_team = input("Home Team: ")
    away_team = input("Away Team: ")

    if home_team in teams_worldcup and away_team in teams_worldcup:
        break
    

predict_df = get_prediction_df(home_team, away_team)
result_rg_log = search_rg_pol.best_estimator_.predict(predict_df)
result_rg_std = search_rg_std.best_estimator_.predict(predict_df)
result_svm_log = search_svm_pol.best_estimator_.predict(predict_df)
result_svm_std = search_svm_std.best_estimator_.predict(predict_df)


winner_rg_log = home_team if result_rg_log else away_team
winner_rg_std = home_team if result_rg_std else away_team
winner_svm_log = home_team if result_svm_log else away_team
winner_svm_std = home_team if result_svm_std else away_team

print(f"linear regression log -> {home_team} VS. {away_team} winner is: {winner_rg_log} -- {result_rg_log}")
print(f"linear regression std -> {home_team} VS. {away_team} winner is: {winner_rg_std} -- {result_rg_std}")

print(f"SVM log-> {home_team} VS. {away_team} winner is: {winner_svm_log} -- {result_svm_log}")
print(f"SVM std -> {home_team} VS. {away_team} winner is: {winner_svm_std} -- {result_svm_std}")